# ODI to Databricks Migration — SIL_InventoryProductDimension (W_INVENTORY_PRODUCT_D)

**Source ODI Session:** SILOS_SIL_INVENTORYPRODUCTDIMENSION

**Target Table:** `workspace.prxbi_dw.w_inventory_product_d`

**Description:** Incremental load of Inventory Product Dimension with SCD Type 1 logic, dimension lookups, and domain member mapping.

**KM:** IKM BIAPPS Oracle Incremental Update — Detection Strategy: OUTER

In [ ]:
# Cell 1 — Create ETL Parameter Widgets (SCEN_TASK_NO {1} params)
dbutils.widgets.text("DATASOURCE_NUM_ID", "")
dbutils.widgets.text("WH_DATASOURCE_NUM_ID", "")
dbutils.widgets.text("ETL_PROC_WID", "")
dbutils.widgets.text("ODI_SESS_NO", "")
dbutils.widgets.text("ETL_USAGE_CODE", "")
dbutils.widgets.text("IS_INCREMENTAL", "")
dbutils.widgets.text("PRUNE_DAYS", "")
dbutils.widgets.text("EXECUTION_ID", "")
dbutils.widgets.text("LOW_DATE", "")
dbutils.widgets.text("SOURCE_CODE", "")
dbutils.widgets.text("TARGET_CODE", "")

## ETL Parameters

In [ ]:
# Cell 3 — Display ETL parameter values
display(spark.sql("""
SELECT
  '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
  '${WH_DATASOURCE_NUM_ID}' AS WH_DATASOURCE_NUM_ID,
  '${ETL_PROC_WID}' AS ETL_PROC_WID,
  '${ODI_SESS_NO}' AS ODI_SESS_NO,
  '${ETL_USAGE_CODE}' AS ETL_USAGE_CODE,
  '${IS_INCREMENTAL}' AS IS_INCREMENTAL,
  '${PRUNE_DAYS}' AS PRUNE_DAYS,
  '${EXECUTION_ID}' AS EXECUTION_ID,
  '${LOW_DATE}' AS LOW_DATE,
  '${SOURCE_CODE}' AS SOURCE_CODE,
  '${TARGET_CODE}' AS TARGET_CODE
"""))

## SCEN_TASK_NO {1} — Check ETL Load Dates

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {1}: Check if ETL load date entry exists
SELECT
  (CASE
    WHEN COUNT(*) > 0 THEN 'Y'
    ELSE 'N'
  END) AS ETL_LOAD_EXISTS
FROM workspace.prxbi_dw.w_etl_load_dates
WHERE PACKAGE_NAME = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'
  AND (DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID}
       OR DATASOURCE_NUM_ID = ${WH_DATASOURCE_NUM_ID})
  AND ETL_USAGE_CODE = '${ETL_USAGE_CODE}'
  AND COMMITTED = '1';

## SCEN_TASK_NO {2} — MERGE Inventory Product Category Update

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {2}: Update inv_prod_cat1 and inv_prod_cat1_wid on w_inventory_product_d
MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T
USING (
    SELECT DISTINCT
        x.integration_id,
        x.inv_prod_cat1,
        y.inv_prod_cat1_wid
    FROM
        (
            SELECT
                b.integration_id,
                a.integration_id AS inv_prod_cat1
            FROM
                workspace.prxbi_dw.w_ora_invitem_category_tmp a,
                workspace.prxbi_dw.w_inventory_product_d b
            WHERE
                CONCAT(a.inventory_item_id, '~', a.organization_id) = b.integration_id
                AND a.integration_id <> b.inv_prod_cat1
        ) x,
        (
            SELECT
                p.integration_id,
                q.row_wid AS inv_prod_cat1_wid
            FROM
                workspace.prxbi_dw.w_ora_invitem_category_tmp p,
                workspace.prxbi_dw.w_prod_cat_dh q
            WHERE
                q.integration_id = p.integration_id
        ) y
    WHERE
        x.inv_prod_cat1 = y.integration_id
) AS S
ON T.integration_id = S.integration_id
WHEN MATCHED THEN UPDATE SET
    T.inv_prod_cat1 = S.inv_prod_cat1,
    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid;

## SCEN_TASK_NO {30}-{50} — IKM BIAPPS Incremental Update Setup

SCEN_TASK_NO {10}, {20}, {30}: KM options and comments (no-op)

SCEN_TASK_NO {40}, {50}: PL/SQL BEGIN/END blocks — removed (not applicable in Spark)

## SCEN_TASK_NO {60}-{70} — Error Table

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}:
Drop error table
DROP TABLE IF EXISTS workspace.prxbi_dw.e_3260538_1;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {70}: Create error table
CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.e_3260538_1
(
    ORA_ERR_NUMBER$ BIGINT,
    ORA_ERR_MESG$ STRING,
    ORA_ERR_ROWID$ STRING,
    ORA_ERR_OPTYP$ STRING,
    ORA_ERR_TAG$ STRING,
    IND_UPDATE STRING,
    DIAGNOSTIC_ROWID STRING,
    ERROR_TYPE_IND STRING,
    AUTOCORRECT_IND STRING DEFAULT 'N',
    AUTOCORRECT_CODE STRING,
    AUTOCORRECT_DESC STRING,
    COMMITTED STRING DEFAULT '0',
    ROW_WID STRING,
    PRODUCT_WID STRING,
    INVENTORY_ORG_WID STRING,
    PLANT_LOC_WID STRING,
    PRODUCT_NUM STRING,
    ABC_IND STRING,
    PLANNER_CODE STRING,
    PROCUREMENT_TYPE_CODE STRING,
    SPC_PROC_TYPE_CODE STRING,
    BUYER_CODE STRING,
    BUYER_NAME STRING,
    COMMODITY_CODE STRING,
    COMMODITY_UOM_CODE STRING,
    PROFIT_CENTER_NUM STRING,
    REORDER_POINT STRING,
    SAFETY_STOCK_LEVEL STRING,
    MIN_LOT_SIZE STRING,
    MAX_LOT_SIZE STRING,
    FIXED_LOT_SIZE STRING,
    MAX_STOCK_LEVEL STRING,
    LOT_ORDERING_COST STRING,
    MRP_TIME_FENCE STRING,
    EXT_PROCURE_TIME STRING,
    INTERNAL_MFG_TIME STRING,
    MAX_STORAGE_DAYS STRING,
    MRP_PROFILE_CODE STRING,
    MRP_TYPE_CODE STRING,
    MRP_GRP_CODE STRING,
    LOT_SIZE_CODE STRING,
    BACKFLUSH_IND STRING,
    QA_INSPECT_IND STRING,
    REPETITIVE_MFG_IND STRING,
    BULK_ITEM_IND STRING,
    FORECAST_PERIOD STRING,
    MFG_UOM_CODE STRING,
    ISSUE_UOM_CODE STRING,
    MANUFACTURING_PLACE STRING,
    LOADING_TYPE_CODE STRING,
    INT_STORE_LOC_CODE STRING,
    EXT_STORE_LOC_CODE STRING,
    ACTIVE_FLG STRING,
    CREATED_BY_WID STRING,
    CHANGED_BY_WID STRING,
    CREATED_ON_DT STRING,
    CHANGED_ON_DT STRING,
    AUX1_CHANGED_ON_DT STRING,
    AUX2_CHANGED_ON_DT STRING,
    AUX3_CHANGED_ON_DT STRING,
    AUX4_CHANGED_ON_DT STRING,
    SRC_EFF_FROM_DT STRING,
    SRC_EFF_TO_DT STRING,
    EFFECTIVE_FROM_DT STRING,
    EFFECTIVE_TO_DT STRING,
    CURRENT_FLG STRING,
    W_INSERT_DT STRING,
    W_UPDATE_DT STRING,
    DATASOURCE_NUM_ID STRING,
    ETL_PROC_WID STRING,
    INTEGRATION_ID STRING,
    TENANT_ID STRING,
    X_CUSTOM STRING,
    INV_PROD_CAT1 STRING,
    INV_PROD_CAT2 STRING,
    INV_PROD_CAT3 STRING,
    INV_PROD_CAT4 STRING,
    INV_PROD_CAT5 STRING,
    INV_PROD_CAT6 STRING,
    INV_PROD_CAT7 STRING,
    INV_PROD_CAT8 STRING,
    INV_PROD_CAT9 STRING,
    INV_PROD_CAT10 STRING,
    INV_PROD_CAT1_WID STRING,
    INV_PROD_CAT2_WID STRING,
    INV_PROD_CAT3_WID STRING,
    INV_PROD_CAT4_WID STRING,
    INV_PROD_CAT5_WID STRING,
    INV_PROD_CAT6_WID STRING,
    INV_PROD_CAT7_WID STRING,
    INV_PROD_CAT8_WID STRING,
    INV_PROD_CAT9_WID STRING,
    INV_PROD_CAT10_WID STRING,
    INVOICEABLE_ITEM_FLAG STRING,
    INVOICE_ENABLED_FLAG STRING,
    PRIMARY_UOM_CODE STRING,
    C_PRIMARY_UOM_CODE STRING,
    UNSPSC_CODE STRING,
    UNSPSC_INV_PROD_CAT_WID STRING,
    COMMODITY_NAME STRING,
    COMMODITY_UOM_NAME STRING,
    EXT_STORE_LOC_NAME STRING,
    INT_STORE_LOC_NAME STRING,
    ISSUE_UOM_NAME STRING,
    LOADING_TYPE_NAME STRING,
    LOT_SIZE_NAME STRING,
    MFG_UOM_NAME STRING,
    MRP_GRP_NAME STRING,
    MRP_PROFILE_NAME STRING,
    MRP_TYPE_NAME STRING,
    PLANNER_NAME STRING,
    PRIMARY_UOM_NAME STRING,
    PROCUREMENT_TYPE_NAME STRING,
    PROFIT_CENTER_NAME STRING,
    SPC_PROC_TYPE_NAME STRING,
    STATUS_CODE STRING,
    W_STATUS_CODE STRING,
    PRODUCT_TYPE_CODE STRING,
    MAKE_BUY_IND STRING,
    FIXED_LEAD_TIME STRING,
    VARIABLE_LEAD_TIME STRING,
    CUMULATIVE_TOTAL_LEAD_TIME STRING,
    POSTPROCESSING_LEAD_TIME STRING,
    PREPROCESSING_LEAD_TIME STRING,
    PROCESS_QUALITY_ENABLED_FLG STRING,
    X_PRICE_SEQUENCE STRING,
    X_ORGANIZATION_NAME STRING,
    X_PRODUCT_DESC STRING,
    X_UOM_DESC STRING,
    X_INV_ITEM_FLG STRING,
    X_STOCK_ITEM_FLG STRING,
    X_TRANS_FLG STRING,
    X_REV_FLG STRING,
    X_COST_FLG STRING,
    X_GCOA_ACCT STRING,
    X_GCOA_PROD STRING,
    X_TAX_CAT STRING,
    ORGANIZATION_ID STRING,
    X_GCOA_LOC_ACCT STRING,
    DELETE_FLG STRING
) USING DELTA;

## SCEN_TASK_NO {110}-{120} — Flow Table

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_w_inventory_product_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {120}: Create flow table
CREATE TABLE workspace.prxbi_dw.i_w_inventory_product_d_flow
(
    SRC_EFF_FROM_DT TIMESTAMP,
    DATASOURCE_NUM_ID BIGINT,
    INTEGRATION_ID STRING,
    ROW_WID DOUBLE,
    PRODUCT_WID DOUBLE,
    INVENTORY_ORG_WID DOUBLE,
    PLANT_LOC_WID DOUBLE,
    PRODUCT_NUM STRING,
    ABC_IND STRING,
    PLANNER_CODE STRING,
    PROCUREMENT_TYPE_CODE STRING,
    SPC_PROC_TYPE_CODE STRING,
    BUYER_CODE STRING,
    BUYER_NAME STRING,
    COMMODITY_CODE STRING,
    COMMODITY_UOM_CODE STRING,
    PROFIT_CENTER_NUM STRING,
    REORDER_POINT DOUBLE,
    SAFETY_STOCK_LEVEL DOUBLE,
    MIN_LOT_SIZE DOUBLE,
    MAX_LOT_SIZE DOUBLE,
    FIXED_LOT_SIZE DOUBLE,
    MAX_STOCK_LEVEL DOUBLE,
    LOT_ORDERING_COST DOUBLE,
    MRP_TIME_FENCE DOUBLE,
    EXT_PROCURE_TIME DOUBLE,
    INTERNAL_MFG_TIME DOUBLE,
    MAX_STORAGE_DAYS DOUBLE,
    MRP_PROFILE_CODE STRING,
    MRP_TYPE_CODE STRING,
    MRP_GRP_CODE STRING,
    LOT_SIZE_CODE STRING,
    BACKFLUSH_IND STRING,
    QA_INSPECT_IND STRING,
    REPETITIVE_MFG_IND STRING,
    BULK_ITEM_IND STRING,
    FORECAST_PERIOD STRING,
    MFG_UOM_CODE STRING,
    ISSUE_UOM_CODE STRING,
    MANUFACTURING_PLACE STRING,
    LOADING_TYPE_CODE STRING,
    INT_STORE_LOC_CODE STRING,
    EXT_STORE_LOC_CODE STRING,
    ACTIVE_FLG STRING,
    CREATED_BY_WID DOUBLE,
    CHANGED_BY_WID DOUBLE,
    CREATED_ON_DT TIMESTAMP,
    CHANGED_ON_DT TIMESTAMP,
    AUX1_CHANGED_ON_DT TIMESTAMP,
    AUX2_CHANGED_ON_DT TIMESTAMP,
    AUX3_CHANGED_ON_DT TIMESTAMP,
    AUX4_CHANGED_ON_DT TIMESTAMP,
    SRC_EFF_TO_DT TIMESTAMP,
    EFFECTIVE_FROM_DT TIMESTAMP,
    EFFECTIVE_TO_DT TIMESTAMP,
    DELETE_FLG STRING,
    CURRENT_FLG STRING,
    W_INSERT_DT TIMESTAMP,
    W_UPDATE_DT TIMESTAMP,
    ETL_PROC_WID DOUBLE,
    TENANT_ID STRING,
    X_CUSTOM STRING,
    INV_PROD_CAT1 STRING,
    INV_PROD_CAT2 STRING,
    INV_PROD_CAT3 STRING,
    INV_PROD_CAT4 STRING,
    INV_PROD_CAT5 STRING,
    INV_PROD_CAT6 STRING,
    INV_PROD_CAT7 STRING,
    INV_PROD_CAT8 STRING,
    INV_PROD_CAT9 STRING,
    INV_PROD_CAT10 STRING,
    INV_PROD_CAT1_WID DOUBLE,
    INV_PROD_CAT2_WID DOUBLE,
    INV_PROD_CAT3_WID DOUBLE,
    INV_PROD_CAT4_WID DOUBLE,
    INV_PROD_CAT5_WID DOUBLE,
    INV_PROD_CAT6_WID DOUBLE,
    INV_PROD_CAT7_WID DOUBLE,
    INV_PROD_CAT8_WID DOUBLE,
    INV_PROD_CAT9_WID DOUBLE,
    INV_PROD_CAT10_WID DOUBLE,
    INVOICEABLE_ITEM_FLAG STRING,
    INVOICE_ENABLED_FLAG STRING,
    PRIMARY_UOM_CODE STRING,
    C_PRIMARY_UOM_CODE STRING,
    UNSPSC_CODE STRING,
    UNSPSC_INV_PROD_CAT_WID DOUBLE,
    COMMODITY_NAME STRING,
    COMMODITY_UOM_NAME STRING,
    EXT_STORE_LOC_NAME STRING,
    INT_STORE_LOC_NAME STRING,
    ISSUE_UOM_NAME STRING,
    LOADING_TYPE_NAME STRING,
    LOT_SIZE_NAME STRING,
    MFG_UOM_NAME STRING,
    MRP_GRP_NAME STRING,
    MRP_PROFILE_NAME STRING,
    MRP_TYPE_NAME STRING,
    PLANNER_NAME STRING,
    PRIMARY_UOM_NAME STRING,
    PROCUREMENT_TYPE_NAME STRING,
    PROFIT_CENTER_NAME STRING,
    SPC_PROC_TYPE_NAME STRING,
    STATUS_CODE STRING,
    W_STATUS_CODE STRING,
    PRODUCT_TYPE_CODE STRING,
    MAKE_BUY_IND STRING,
    FIXED_LEAD_TIME DOUBLE,
    VARIABLE_LEAD_TIME DOUBLE,
    CUMULATIVE_TOTAL_LEAD_TIME DOUBLE,
    POSTPROCESSING_LEAD_TIME DOUBLE,
    PREPROCESSING_LEAD_TIME DOUBLE,
    PROCESS_QUALITY_ENABLED_FLG STRING,
    X_PRICE_SEQUENCE STRING,
    X_ORGANIZATION_NAME STRING,
    X_PRODUCT_DESC STRING,
    X_UOM_DESC STRING,
    X_INV_ITEM_FLG STRING,
    X_STOCK_ITEM_FLG STRING,
    X_TRANS_FLG STRING,
    X_REV_FLG STRING,
    X_COST_FLG STRING,
    X_GCOA_ACCT STRING,
    X_GCOA_PROD STRING,
    X_TAX_CAT STRING,
    ORGANIZATION_ID STRING,
    X_GCOA_LOC_ACCT STRING,
    IND_UPDATE STRING
) USING DELTA;

## SCEN_TASK_NO {130} — Insert into Flow Table (Detection Strategy: OUTER)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Insert into flow table with dimension lookups and change detection
-- Detection Strategy: OUTER — IND_UPDATE derived via LEFT JOIN to target
-- Converted: NVL->COALESCE, ||->CONCAT, TO_DATE->to_date, ROWID->target join existence, SYSDATE->current_timestamp()
INSERT INTO workspace.prxbi_dw.i_w_inventory_product_d_flow
(
    PRODUCT_WID,
    INVENTORY_ORG_WID,
    PLANT_LOC_WID,
    PRODUCT_NUM,
    ABC_IND,
    PLANNER_CODE,
    PROCUREMENT_TYPE_CODE,
    SPC_PROC_TYPE_CODE,
    BUYER_CODE,
    BUYER_NAME,
    COMMODITY_CODE,
    COMMODITY_UOM_CODE,
    PROFIT_CENTER_NUM,
    REORDER_POINT,
    SAFETY_STOCK_LEVEL,
    MIN_LOT_SIZE,
    MAX_LOT_SIZE,
    FIXED_LOT_SIZE,
    MAX_STOCK_LEVEL,
    LOT_ORDERING_COST,
    MRP_TIME_FENCE,
    EXT_PROCURE_TIME,
    INTERNAL_MFG_TIME,
    MAX_STORAGE_DAYS,
    MRP_PROFILE_CODE,
    MRP_TYPE_CODE,
    MRP_GRP_CODE,
    LOT_SIZE_CODE,
    BACKFLUSH_IND,
    QA_INSPECT_IND,
    REPETITIVE_MFG_IND,
    BULK_ITEM_IND,
    FORECAST_PERIOD,
    MFG_UOM_CODE,
    ISSUE_UOM_CODE,
    MANUFACTURING_PLACE,
    LOADING_TYPE_CODE,
    INT_STORE_LOC_CODE,
    EXT_STORE_LOC_CODE,
    ACTIVE_FLG,
    CREATED_BY_WID,
    CHANGED_BY_WID,
    CREATED_ON_DT,
    CHANGED_ON_DT,
    AUX1_CHANGED_ON_DT,
    AUX2_CHANGED_ON_DT,
    AUX3_CHANGED_ON_DT,
    AUX4_CHANGED_ON_DT,
    SRC_EFF_FROM_DT,
    SRC_EFF_TO_DT,
    EFFECTIVE_FROM_DT,
    DELETE_FLG,
    DATASOURCE_NUM_ID,
    INTEGRATION_ID,
    TENANT_ID,
    X_CUSTOM,
    INV_PROD_CAT1,
    INV_PROD_CAT2,
    INV_PROD_CAT3,
    INV_PROD_CAT4,
    INV_PROD_CAT5,
    INV_PROD_CAT6,
    INV_PROD_CAT7,
    INV_PROD_CAT8,
    INV_PROD_CAT9,
    INV_PROD_CAT10,
    INV_PROD_CAT1_WID,
    INV_PROD_CAT2_WID,
    INV_PROD_CAT3_WID,
    INV_PROD_CAT4_WID,
    INV_PROD_CAT5_WID,
    INV_PROD_CAT6_WID,
    INV_PROD_CAT7_WID,
    INV_PROD_CAT8_WID,
    INV_PROD_CAT9_WID,
    INV_PROD_CAT10_WID,
    INVOICEABLE_ITEM_FLAG,
    INVOICE_ENABLED_FLAG,
    PRIMARY_UOM_CODE,
    C_PRIMARY_UOM_CODE,
    UNSPSC_CODE,
    UNSPSC_INV_PROD_CAT_WID,
    COMMODITY_NAME,
    COMMODITY_UOM_NAME,
    EXT_STORE_LOC_NAME,
    INT_STORE_LOC_NAME,
    ISSUE_UOM_NAME,
    LOADING_TYPE_NAME,
    LOT_SIZE_NAME,
    MFG_UOM_NAME,
    MRP_GRP_NAME,
    MRP_PROFILE_NAME,
    MRP_TYPE_NAME,
    PLANNER_NAME,
    PRIMARY_UOM_NAME,
    PROCUREMENT_TYPE_NAME,
    PROFIT_CENTER_NAME,
    SPC_PROC_TYPE_NAME,
    STATUS_CODE,
    W_STATUS_CODE,
    PRODUCT_TYPE_CODE,
    MAKE_BUY_IND,
    FIXED_LEAD_TIME,
    VARIABLE_LEAD_TIME,
    CUMULATIVE_TOTAL_LEAD_TIME,
    POSTPROCESSING_LEAD_TIME,
    PREPROCESSING_LEAD_TIME,
    PROCESS_QUALITY_ENABLED_FLG,
    X_PRICE_SEQUENCE,
    X_ORGANIZATION_NAME,
    X_PRODUCT_DESC,
    X_UOM_DESC,
    X_INV_ITEM_FLG,
    X_STOCK_ITEM_FLG,
    X_TRANS_FLG,
    X_REV_FLG,
    X_COST_FLG,
    X_GCOA_ACCT,
    X_GCOA_PROD,
    X_TAX_CAT,
    ORGANIZATION_ID,
    X_GCOA_LOC_ACCT,
    CURRENT_FLG,
    EFFECTIVE_TO_DT,
    IND_UPDATE
)
SELECT
    C.PRODUCT_WID,
    C.INVENTORY_ORG_WID,
    C.PLANT_LOC_WID,
    C.PRODUCT_NUM,
    C.ABC_IND,
    C.PLANNER_CODE,
    C.PROCUREMENT_TYPE_CODE,
    C.SPC_PROC_TYPE_CODE,
    C.BUYER_CODE,
    C.BUYER_NAME,
    C.COMMODITY_CODE,
    C.COMMODITY_UOM_CODE,
    C.PROFIT_CENTER_NUM,
    C.REORDER_POINT,
    C.SAFETY_STOCK_LEVEL,
    C.MIN_LOT_SIZE,
    C.MAX_LOT_SIZE,
    C.FIXED_LOT_SIZE,
    C.MAX_STOCK_LEVEL,
    C.LOT_ORDERING_COST,
    C.MRP_TIME_FENCE,
    C.EXT_PROCURE_TIME,
    C.INTERNAL_MFG_TIME,
    C.MAX_STORAGE_DAYS,
    C.MRP_PROFILE_CODE,
    C.MRP_TYPE_CODE,
    C.MRP_GRP_CODE,
    C.LOT_SIZE_CODE,
    C.BACKFLUSH_IND,
    C.QA_INSPECT_IND,
    C.REPETITIVE_MFG_IND,
    C.BULK_ITEM_IND,
    C.FORECAST_PERIOD,
    C.MFG_UOM_CODE,
    C.ISSUE_UOM_CODE,
    C.MANUFACTURING_PLACE,
    C.LOADING_TYPE_CODE,
    C.INT_STORE_LOC_CODE,
    C.EXT_STORE_LOC_CODE,
    C.ACTIVE_FLG,
    C.CREATED_BY_WID,
    C.CHANGED_BY_WID,
    C.CREATED_ON_DT,
    C.CHANGED_ON_DT,
    C.AUX1_CHANGED_ON_DT,
    C.AUX2_CHANGED_ON_DT,
    C.AUX3_CHANGED_ON_DT,
    C.AUX4_CHANGED_ON_DT,
    C.SRC_EFF_FROM_DT,
    C.SRC_EFF_TO_DT,
    C.EFFECTIVE_FROM_DT,
    C.DELETE_FLG,
    C.DATASOURCE_NUM_ID,
    C.INTEGRATION_ID,
    C.TENANT_ID,
    C.X_CUSTOM,
    C.INV_PROD_CAT1,
    C.INV_PROD_CAT2,
    C.INV_PROD_CAT3,
    C.INV_PROD_CAT4,
    C.INV_PROD_CAT5,
    C.INV_PROD_CAT6,
    C.INV_PROD_CAT7,
    C.INV_PROD_CAT8,
    C.INV_PROD_CAT9,
    C.INV_PROD_CAT10,
    C.INV_PROD_CAT1_WID,
    C.INV_PROD_CAT2_WID,
    C.INV_PROD_CAT3_WID,
    C.INV_PROD_CAT4_WID,
    C.INV_PROD_CAT5_WID,
    C.INV_PROD_CAT6_WID,
    C.INV_PROD_CAT7_WID,
    C.INV_PROD_CAT8_WID,
    C.INV_PROD_CAT9_WID,
    C.INV_PROD_CAT10_WID,
    C.INVOICEABLE_ITEM_FLAG,
    C.INVOICE_ENABLED_FLAG,
    C.PRIMARY_UOM_CODE,
    C.C_PRIMARY_UOM_CODE,
    C.UNSPSC_CODE,
    C.UNSPSC_INV_PROD_CAT_WID,
    C.COMMODITY_NAME,
    C.COMMODITY_UOM_NAME,
    C.EXT_STORE_LOC_NAME,
    C.INT_STORE_LOC_NAME,
    C.ISSUE_UOM_NAME,
    C.LOADING_TYPE_NAME,
    C.LOT_SIZE_NAME,
    C.MFG_UOM_NAME,
    C.MRP_GRP_NAME,
    C.MRP_PROFILE_NAME,
    C.MRP_TYPE_NAME,
    C.PLANNER_NAME,
    C.PRIMARY_UOM_NAME,
    C.PROCUREMENT_TYPE_NAME,
    C.PROFIT_CENTER_NAME,
    C.SPC_PROC_TYPE_NAME,
    C.STATUS_CODE,
    C.W_STATUS_CODE,
    C.PRODUCT_TYPE_CODE,
    C.MAKE_BUY_IND,
    C.FIXED_LEAD_TIME,
    C.VARIABLE_LEAD_TIME,
    C.CUMULATIVE_TOTAL_LEAD_TIME,
    C.POSTPROCESSING_LEAD_TIME,
    C.PREPROCESSING_LEAD_TIME,
    C.PROCESS_QUALITY_ENABLED_FLG,
    C.X_PRICE_SEQUENCE,
    C.X_ORGANIZATION_NAME,
    C.X_PRODUCT_DESC,
    C.X_UOM_DESC,
    C.X_INV_ITEM_FLG,
    C.X_STOCK_ITEM_FLG,
    C.X_TRANS_FLG,
    C.X_REV_FLG,
    C.X_COST_FLG,
    C.X_GCOA_ACCT,
    C.X_GCOA_PROD,
    C.X_TAX_CAT,
    C.ORGANIZATION_ID,
    C.X_GCOA_LOC_ACCT,
    'Y' AS CURRENT_FLG,
    to_date('01/01/3714 00:00:00', 'MM/dd/yyyy HH:mm:ss') AS EFFECTIVE_TO_DT,
    CASE
        WHEN T.INTEGRATION_ID IS NOT NULL
            AND (T.CHANGED_ON_DT = C.CHANGED_ON_DT OR (T.CHANGED_ON_DT IS NULL AND C.CHANGED_ON_DT IS NULL))
            AND (T.AUX1_CHANGED_ON_DT = C.AUX1_CHANGED_ON_DT OR (T.AUX1_CHANGED_ON_DT IS NULL AND C.AUX1_CHANGED_ON_DT IS NULL))
            AND (T.AUX2_CHANGED_ON_DT = C.AUX2_CHANGED_ON_DT OR (T.AUX2_CHANGED_ON_DT IS NULL AND C.AUX2_CHANGED_ON_DT IS NULL))
            AND (T.AUX3_CHANGED_ON_DT = C.AUX3_CHANGED_ON_DT OR (T.AUX3_CHANGED_ON_DT IS NULL AND C.AUX3_CHANGED_ON_DT IS NULL))
            AND (T.AUX4_CHANGED_ON_DT = C.AUX4_CHANGED_ON_DT OR (T.AUX4_CHANGED_ON_DT IS NULL AND C.AUX4_CHANGED_ON_DT IS NULL))
        THEN 'N'
        WHEN T.INTEGRATION_ID IS NOT NULL
        THEN 'U'
        ELSE 'I'
    END AS IND_UPDATE
FROM
(
    SELECT
        COALESCE(INLINE_VIEW.SCD1_WID_1, 0) AS PRODUCT_WID,
        COALESCE(INLINE_VIEW.SCD1_WID, 0) AS INVENTORY_ORG_WID,
        COALESCE(INLINE_VIEW.ROW_WID, 0) AS PLANT_LOC_WID,
        INLINE_VIEW.PRODUCT_NUM AS PRODUCT_NUM,
        INLINE_VIEW.ABC_IND AS ABC_IND,
        COALESCE(INLINE_VIEW.PLANNER_CODE, '__NOT_APPLICABLE__') AS PLANNER_CODE,
        COALESCE(INLINE_VIEW.PROCUREMENT_TYPE_CODE, '__NOT_APPLICABLE__') AS PROCUREMENT_TYPE_CODE,
        COALESCE(INLINE_VIEW.SPC_PROC_TYPE_CODE, '__NOT_APPLICABLE__') AS SPC_PROC_TYPE_CODE,
        COALESCE(INLINE_VIEW.BUYER_CODE, '__NOT_APPLICABLE__') AS BUYER_CODE,
        INLINE_VIEW.BUYER_NAME AS BUYER_NAME,
        COALESCE(INLINE_VIEW.COMMODITY_CODE, '__NOT_APPLICABLE__') AS COMMODITY_CODE,
        COALESCE(INLINE_VIEW.COMMODITY_UOM_CODE, '__NOT_APPLICABLE__') AS COMMODITY_UOM_CODE,
        INLINE_VIEW.PROFIT_CENTER_NUM AS PROFIT_CENTER_NUM,
        INLINE_VIEW.REORDER_POINT AS REORDER_POINT,
        INLINE_VIEW.SAFETY_STOCK_LEVEL AS SAFETY_STOCK_LEVEL,
        INLINE_VIEW.MIN_LOT_SIZE AS MIN_LOT_SIZE,
        INLINE_VIEW.MAX_LOT_SIZE AS MAX_LOT_SIZE,
        INLINE_VIEW.FIXED_LOT_SIZE AS FIXED_LOT_SIZE,
        INLINE_VIEW.MAX_STOCK_LEVEL AS MAX_STOCK_LEVEL,
        INLINE_VIEW.LOT_ORDERING_COST AS LOT_ORDERING_COST,
        INLINE_VIEW.MRP_TIME_FENCE AS MRP_TIME_FENCE,
        INLINE_VIEW.EXT_PROCURE_TIME AS EXT_PROCURE_TIME,
        INLINE_VIEW.INTERNAL_MFG_TIME AS INTERNAL_MFG_TIME,
        INLINE_VIEW.MAX_STORAGE_DAYS AS MAX_STORAGE_DAYS,
        COALESCE(INLINE_VIEW.MRP_PROFILE_CODE, '__NOT_APPLICABLE__') AS MRP_PROFILE_CODE,
        COALESCE(INLINE_VIEW.MRP_TYPE_CODE, '__NOT_APPLICABLE__') AS MRP_TYPE_CODE,
        COALESCE(INLINE_VIEW.MRP_GRP_CODE, '__NOT_APPLICABLE__') AS MRP_GRP_CODE,
        COALESCE(INLINE_VIEW.LOT_SIZE_CODE, '__NOT_APPLICABLE__') AS LOT_SIZE_CODE,
        INLINE_VIEW.BACKFLUSH_IND AS BACKFLUSH_IND,
        INLINE_VIEW.QA_INSPECT_IND AS QA_INSPECT_IND,
        INLINE_VIEW.REPETITIVE_MFG_IND AS REPETITIVE_MFG_IND,
        INLINE_VIEW.BULK_ITEM_IND AS BULK_ITEM_IND,
        INLINE_VIEW.FORECAST_PERIOD AS FORECAST_PERIOD,
        COALESCE(INLINE_VIEW.MFG_UOM_CODE, '__NOT_APPLICABLE__') AS MFG_UOM_CODE,
        COALESCE(INLINE_VIEW.ISSUE_UOM_CODE, '__NOT_APPLICABLE__') AS ISSUE_UOM_CODE,
        INLINE_VIEW.MANUFACTURING_PLACE AS MANUFACTURING_PLACE,
        COALESCE(INLINE_VIEW.LOADING_TYPE_CODE, '__NOT_APPLICABLE__') AS LOADING_TYPE_CODE,
        COALESCE(INLINE_VIEW.INT_STORE_LOC_CODE, '__NOT_APPLICABLE__') AS INT_STORE_LOC_CODE,
        COALESCE(INLINE_VIEW.EXT_STORE_LOC_CODE, '__NOT_APPLICABLE__') AS EXT_STORE_LOC_CODE,
        INLINE_VIEW.ACTIVE_FLG AS ACTIVE_FLG,
        COALESCE(LKP_W_USER_D_LKP_W_USER_D_CR_1.ROW_WID, 0) AS CREATED_BY_WID,
        COALESCE(INLINE_VIEW.ROW_WID_1, 0) AS CHANGED_BY_WID,
        INLINE_VIEW.CREATED_ON_DT AS CREATED_ON_DT,
        INLINE_VIEW.CHANGED_ON_DT AS CHANGED_ON_DT,
        INLINE_VIEW.AUX1_CHANGED_ON_DT AS AUX1_CHANGED_ON_DT,
        INLINE_VIEW.AUX2_CHANGED_ON_DT AS AUX2_CHANGED_ON_DT,
        INLINE_VIEW.AUX3_CHANGED_ON_DT AS AUX3_CHANGED_ON_DT,
        INLINE_VIEW.AUX4_CHANGED_ON_DT AS AUX4_CHANGED_ON_DT,
        INLINE_VIEW.SRC_EFF_FROM_DT AS SRC_EFF_FROM_DT,
        INLINE_VIEW.SRC_EFF_TO_DT AS SRC_EFF_TO_DT,
        COALESCE(INLINE_VIEW.SRC_EFF_FROM_DT, to_timestamp(substring('${LOW_DATE}', 1, 19), 'yyyy-MM-dd HH:mm:ss')) AS EFFECTIVE_FROM_DT,
        (CASE WHEN INLINE_VIEW.DELETE_FLG = 'Y' THEN 'Y' ELSE 'N' END) AS DELETE_FLG,
        INLINE_VIEW.DATASOURCE_NUM_ID AS DATASOURCE_NUM_ID,
        INLINE_VIEW.INTEGRATION_ID AS INTEGRATION_ID,
        INLINE_VIEW.TENANT_ID AS TENANT_ID,
        INLINE_VIEW.X_CUSTOM AS X_CUSTOM,
        INLINE_VIEW.INV_PROD_CAT1 AS INV_PROD_CAT1,
        INLINE_VIEW.INV_PROD_CAT2 AS INV_PROD_CAT2,
        INLINE_VIEW.INV_PROD_CAT3 AS INV_PROD_CAT3,
        INLINE_VIEW.INV_PROD_CAT4 AS INV_PROD_CAT4,
        INLINE_VIEW.INV_PROD_CAT5 AS INV_PROD_CAT5,
        INLINE_VIEW.INV_PROD_CAT6 AS INV_PROD_CAT6,
        INLINE_VIEW.INV_PROD_CAT7 AS INV_PROD_CAT7,
        INLINE_VIEW.INV_PROD_CAT8 AS INV_PROD_CAT8,
        INLINE_VIEW.INV_PROD_CAT9 AS INV_PROD_CAT9,
        INLINE_VIEW.INV_PROD_CAT10 AS INV_PROD_CAT10,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT1 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT1_ROW_WID, 0) END) AS INV_PROD_CAT1_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT2 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT2_ROW_WID, 0) END) AS INV_PROD_CAT2_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT3 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT3_ROW_WID, 0) END) AS INV_PROD_CAT3_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT4 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT4_ROW_WID, 0) END) AS INV_PROD_CAT4_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT5 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT5_ROW_WID, 0) END) AS INV_PROD_CAT5_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT6 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT6_ROW_WID, 0) END) AS INV_PROD_CAT6_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT7 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT7_ROW_WID, 0) END) AS INV_PROD_CAT7_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT8 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT8_ROW_WID, 0) END) AS INV_PROD_CAT8_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT9 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT9_ROW_WID, 0) END) AS INV_PROD_CAT9_WID,
        (CASE WHEN INLINE_VIEW.INV_PROD_CAT10 IS NULL THEN 0 ELSE COALESCE(INLINE_VIEW.INV_PROD_CAT10_ROW_WID, 0) END) AS INV_PROD_CAT10_WID,
        COALESCE(INLINE_VIEW.INVOICEABLE_ITEM_FLAG, 'N') AS INVOICEABLE_ITEM_FLAG,
        COALESCE(INLINE_VIEW.INVOICE_ENABLED_FLAG, 'N') AS INVOICE_ENABLED_FLAG,
        COALESCE(INLINE_VIEW.PRIMARY_UOM_CODE, '__NOT_APPLICABLE__') AS PRIMARY_UOM_CODE,
        COALESCE(
            (
                SELECT W_DOMAIN_MEMBER_MAP_G.TRG_DOMAIN_MEMBER_CODE
                FROM workspace.prxbi_dw.w_domain_member_map_g W_DOMAIN_MEMBER_MAP_G
                WHERE W_DOMAIN_MEMBER_MAP_G.SRC_DOMAIN_CODE = '${SOURCE_CODE}'
                  AND W_DOMAIN_MEMBER_MAP_G.SRC_DOMAIN_MEMBER_CODE = COALESCE(INLINE_VIEW.PRIMARY_UOM_CODE, '__UNASSIGNED__')
                  AND W_DOMAIN_MEMBER_MAP_G.SRC_DATASOURCE_NUM_ID IN (INLINE_VIEW.DATASOURCE_NUM_ID, 999)
                  AND W_DOMAIN_MEMBER_MAP_G.TRG_DOMAIN_CODE = '${TARGET_CODE}'
            ),
            (
                SELECT W_DOMAIN_MEMBER_MAP_G.TRG_DOMAIN_MEMBER_CODE
                FROM workspace.prxbi_dw.w_domain_member_map_g W_DOMAIN_MEMBER_MAP_G
                WHERE W_DOMAIN_MEMBER_MAP_G.SRC_DOMAIN_CODE = '${SOURCE_CODE}'
                  AND W_DOMAIN_MEMBER_MAP_G.SRC_DOMAIN_MEMBER_CODE = '__ANY__'
                  AND W_DOMAIN_MEMBER_MAP_G.SRC_DATASOURCE_NUM_ID IN (INLINE_VIEW.DATASOURCE_NUM_ID, 999)
                  AND W_DOMAIN_MEMBER_MAP_G.TRG_DOMAIN_CODE = '${TARGET_CODE}'
            ),
            (CASE
                WHEN INLINE_VIEW.PRIMARY_UOM_CODE IS NULL THEN COALESCE(
                    (
                        SELECT W_DOMAIN_MEMBER_G.DOMAIN_MEMBER_CODE
                        FROM workspace.prxbi_dw.w_domain_member_g W_DOMAIN_MEMBER_G
                        WHERE W_DOMAIN_MEMBER_G.DOMAIN_MEMBER_CODE = '__UNASSIGNED__'
                          AND DOMAIN_CODE = '${TARGET_CODE}'
                    ), '__ERROR__')
                ELSE (CASE WHEN '${TARGET_CODE}' = 'W_LANGUAGE' THEN '_ERR' ELSE '__ERROR__' END)
            END)
        ) AS C_PRIMARY_UOM_CODE,
        COALESCE(INLINE_VIEW.UNSPSC_CODE, '__NOT_APPLICABLE__') AS UNSPSC_CODE,
        COALESCE(INLINE_VIEW.INV_PROD_CAT_UNSPSC_ROW_WID, 0) AS UNSPSC_INV_PROD_CAT_WID,
        INLINE_VIEW.COMMODITY_NAME AS COMMODITY_NAME,
        INLINE_VIEW.COMMODITY_UOM_NAME AS COMMODITY_UOM_NAME,
        INLINE_VIEW.EXT_STORE_LOC_NAME AS EXT_STORE_LOC_NAME,
        INLINE_VIEW.INT_STORE_LOC_NAME AS INT_STORE_LOC_NAME,
        INLINE_VIEW.ISSUE_UOM_NAME AS ISSUE_UOM_NAME,
        INLINE_VIEW.LOADING_TYPE_NAME AS LOADING_TYPE_NAME,
        INLINE_VIEW.LOT_SIZE_NAME AS LOT_SIZE_NAME,
        INLINE_VIEW.MFG_UOM_NAME AS MFG_UOM_NAME,
        INLINE_VIEW.MRP_GRP_NAME AS MRP_GRP_NAME,
        INLINE_VIEW.MRP_PROFILE_NAME AS MRP_PROFILE_NAME,
        INLINE_VIEW.MRP_TYPE_NAME AS MRP_TYPE_NAME,
        INLINE_VIEW.PLANNER_NAME AS PLANNER_NAME,
        INLINE_VIEW.PRIMARY_UOM_NAME AS PRIMARY_UOM_NAME,
        INLINE_VIEW.PROCUREMENT_TYPE_NAME AS PROCUREMENT_TYPE_NAME,
        INLINE_VIEW.PROFIT_CENTER_NAME AS PROFIT_CENTER_NAME,
        INLINE_VIEW.SPC_PROC_TYPE_NAME AS SPC_PROC_TYPE_NAME,
        INLINE_VIEW.STATUS_CODE AS STATUS_CODE,
        INLINE_VIEW.W_STATUS_CODE AS W_STATUS_CODE,
        INLINE_VIEW.PRODUCT_TYPE_CODE AS PRODUCT_TYPE_CODE,
        INLINE_VIEW.MAKE_BUY_IND AS MAKE_BUY_IND,
        INLINE_VIEW.FIXED_LEAD_TIME AS FIXED_LEAD_TIME,
        INLINE_VIEW.VARIABLE_LEAD_TIME AS VARIABLE_LEAD_TIME,
        INLINE_VIEW.CUMULATIVE_TOTAL_LEAD_TIME AS CUMULATIVE_TOTAL_LEAD_TIME,
        INLINE_VIEW.PREPROCESSING_LEAD_TIME AS POSTPROCESSING_LEAD_TIME,
        INLINE_VIEW.PREPROCESSING_LEAD_TIME AS PREPROCESSING_LEAD_TIME,
        INLINE_VIEW.PROCESS_QUALITY_ENABLED_FLG AS PROCESS_QUALITY_ENABLED_FLG,
        INLINE_VIEW.X_PRICE_SEQUENCE AS X_PRICE_SEQUENCE,
        INLINE_VIEW.X_ORGANIZATION_NAME AS X_ORGANIZATION_NAME,
        INLINE_VIEW.X_PRODUCT_DESC AS X_PRODUCT_DESC,
        INLINE_VIEW.X_UOM_DESC AS X_UOM_DESC,
        INLINE_VIEW.X_INV_ITEM_FLG AS X_INV_ITEM_FLG,
        INLINE_VIEW.X_STOCK_ITEM_FLG AS X_STOCK_ITEM_FLG,
        INLINE_VIEW.X_TRANS_FLG AS X_TRANS_FLG,
        INLINE_VIEW.X_REV_FLG AS X_REV_FLG,
        INLINE_VIEW.X_COST_FLG AS X_COST_FLG,
        INLINE_VIEW.X_GCOA_ACCT AS X_GCOA_ACCT,
        INLINE_VIEW.X_GCOA_PROD AS X_GCOA_PROD,
        INLINE_VIEW.X_TAX_CAT AS X_TAX_CAT,
        INLINE_VIEW.INVENTORY_ORG_ID AS ORGANIZATION_ID,
        INLINE_VIEW.X_GCOA_LOC_ACCT AS X_GCOA_LOC_ACCT
    FROM (
        SELECT
            SQ_W.MRP_TYPE_NAME, SQ_W.X_CUSTOM, SQ_W.SRC_EFF_TO_DT, SQ_W.EXT_PROCURE_TIME,
            SQ_W.COMMODITY_UOM_CODE, SQ_W.INV_PROD_CAT10, SQ_W.INVOICE_ENABLED_FLAG,
            SQ_W.MRP_PROFILE_CODE, SQ_W.EXT_STORE_LOC_NAME, SQ_W.AUX3_CHANGED_ON_DT,
            SQ_W.MIN_LOT_SIZE, SQ_W.REORDER_POINT, SQ_W.BULK_ITEM_IND, SQ_W.SPC_PROC_TYPE_NAME,
            SQ_W.COMMODITY_CODE, SQ_W.REPETITIVE_MFG_IND, SQ_W.LOADING_TYPE_CODE,
            SQ_W.PROFIT_CENTER_NAME, SQ_W.CREATED_BY_ID, SQ_W.BUYER_CODE,
            SQ_W.INT_STORE_LOC_NAME, SQ_W.COMMODITY_NAME, SQ_W.PLANNER_NAME,
            SQ_W.MAX_STORAGE_DAYS, SQ_W.PLANNER_CODE, SQ_W.MAX_LOT_SIZE,
            SQ_W.MRP_GRP_CODE, SQ_W.LOT_ORDERING_COST, SQ_W.INTERNAL_MFG_TIME,
            SQ_W.MANUFACTURING_PLACE, SQ_W.PROCUREMENT_TYPE_NAME, SQ_W.PLANT_LOC_ID,
            SQ_W.CHANGED_BY_ID, SQ_W.BACKFLUSH_IND, SQ_W.AUX2_CHANGED_ON_DT,
            SQ_W.DATASOURCE_NUM_ID, SQ_W.SPC_PROC_TYPE_CODE, SQ_W.LOT_SIZE_CODE,
            SQ_W.CHANGED_ON_DT, SQ_W.PRODUCT_ID, SQ_W.PRIMARY_UOM_CODE, SQ_W.PRODUCT_NUM,
            SQ_W.FORECAST_PERIOD, SQ_W.AUX1_CHANGED_ON_DT, SQ_W.BUYER_NAME,
            SQ_W.FIXED_LOT_SIZE, SQ_W.MRP_GRP_NAME, SQ_W.PROCUREMENT_TYPE_CODE,
            SQ_W.PRIMARY_UOM_NAME, SQ_W.MAX_STOCK_LEVEL, SQ_W.SAFETY_STOCK_LEVEL,
            SQ_W.INT_STORE_LOC_CODE, SQ_W.EXT_STORE_LOC_CODE, SQ_W.ISSUE_UOM_NAME,
            SQ_W.ABC_IND, SQ_W.PROFIT_CENTER_NUM, SQ_W.CREATED_ON_DT, SQ_W.UNSPSC_CODE,
            SQ_W.INV_PROD_CAT1, SQ_W.INV_PROD_CAT3, SQ_W.INV_PROD_CAT2,
            SQ_W.COMMODITY_UOM_NAME, SQ_W.INV_PROD_CAT5, SQ_W.INV_PROD_CAT4,
            SQ_W.MFG_UOM_NAME, SQ_W.INV_PROD_CAT7, SQ_W.INV_PROD_CAT6,
            SQ_W.INV_PROD_CAT9, SQ_W.INV_PROD_CAT8, SQ_W.AUX4_CHANGED_ON_DT,
            SQ_W.LOT_SIZE_NAME, SQ_W.TENANT_ID, SQ_W.INVOICEABLE_ITEM_FLAG,
            SQ_W.INVENTORY_ORG_ID, SQ_W.SRC_EFF_FROM_DT, SQ_W.INTEGRATION_ID,
            SQ_W.ISSUE_UOM_CODE, SQ_W.DELETE_FLG, SQ_W.MRP_TYPE_CODE, SQ_W.ACTIVE_FLG,
            SQ_W.MRP_TIME_FENCE, SQ_W.QA_INSPECT_IND, SQ_W.MFG_UOM_CODE,
            SQ_W.LOADING_TYPE_NAME, SQ_W.MRP_PROFILE_NAME,
            SQ_W.INV_PROD_CAT1_ROW_WID, SQ_W.INV_PROD_CAT2_ROW_WID,
            SQ_W.INV_PROD_CAT3_ROW_WID, SQ_W.INV_PROD_CAT4_ROW_WID,
            SQ_W.INV_PROD_CAT5_ROW_WID, SQ_W.INV_PROD_CAT6_ROW_WID,
            SQ_W.INV_PROD_CAT7_ROW_WID, SQ_W.INV_PROD_CAT8_ROW_WID,
            SQ_W.INV_PROD_CAT_UNSPSC_ROW_WID, SQ_W.INV_PROD_CAT9_ROW_WID,
            SQ_W.INV_PROD_CAT10_ROW_WID, SQ_W.CUMULATIVE_TOTAL_LEAD_TIME,
            SQ_W.FIXED_LEAD_TIME, SQ_W.W_STATUS_CODE, SQ_W.PREPROCESSING_LEAD_TIME,
            SQ_W.STATUS_CODE, SQ_W.VARIABLE_LEAD_TIME, SQ_W.MAKE_BUY_IND,
            SQ_W.PROCESS_QUALITY_ENABLED_FLG, SQ_W.PRODUCT_TYPE_CODE,
            SQ_W.X_PRICE_SEQUENCE, SQ_W.X_ORGANIZATION_NAME, SQ_W.X_PRODUCT_DESC,
            SQ_W.X_UOM_DESC, SQ_W.X_INV_ITEM_FLG, SQ_W.X_STOCK_ITEM_FLG,
            SQ_W.X_TRANS_FLG, SQ_W.X_REV_FLG, SQ_W.X_COST_FLG,
            SQ_W.X_GCOA_ACCT, SQ_W.X_GCOA_PROD, SQ_W.X_TAX_CAT,
            SQ_W.X_GCOA_LOC_ACCT,
            LKP_BL.ROW_WID AS ROW_WID,
            LKP_IO.SCD1_WID AS SCD1_WID,
            LKP_PD.SCD1_WID AS SCD1_WID_1,
            LKP_UC.ROW_WID AS ROW_WID_1
        FROM
            (
                SELECT
                    DS.MRP_TYPE_NAME, DS.X_CUSTOM, DS.SRC_EFF_TO_DT, DS.EXT_PROCURE_TIME,
                    DS.COMMODITY_UOM_CODE, DS.INV_PROD_CAT10, DS.INVOICE_ENABLED_FLAG,
                    DS.MRP_PROFILE_CODE, DS.EXT_STORE_LOC_NAME, DS.AUX3_CHANGED_ON_DT,
                    DS.MIN_LOT_SIZE, DS.REORDER_POINT, DS.BULK_ITEM_IND, DS.SPC_PROC_TYPE_NAME,
                    DS.COMMODITY_CODE, DS.REPETITIVE_MFG_IND, DS.LOADING_TYPE_CODE,
                    DS.PROFIT_CENTER_NAME, DS.CREATED_BY_ID, DS.BUYER_CODE,
                    DS.INT_STORE_LOC_NAME, DS.COMMODITY_NAME, DS.PLANNER_NAME,
                    DS.MAX_STORAGE_DAYS, DS.PLANNER_CODE, DS.MAX_LOT_SIZE,
                    DS.MRP_GRP_CODE, DS.LOT_ORDERING_COST, DS.INTERNAL_MFG_TIME,
                    DS.MANUFACTURING_PLACE, DS.PROCUREMENT_TYPE_NAME, DS.PLANT_LOC_ID,
                    DS.CHANGED_BY_ID, DS.BACKFLUSH_IND, DS.AUX2_CHANGED_ON_DT,
                    DS.DATASOURCE_NUM_ID, DS.SPC_PROC_TYPE_CODE, DS.LOT_SIZE_CODE,
                    DS.CHANGED_ON_DT, DS.PRODUCT_ID, DS.PRIMARY_UOM_CODE, DS.PRODUCT_NUM,
                    DS.FORECAST_PERIOD, DS.AUX1_CHANGED_ON_DT, DS.BUYER_NAME,
                    DS.FIXED_LOT_SIZE, DS.MRP_GRP_NAME, DS.PROCUREMENT_TYPE_CODE,
                    DS.PRIMARY_UOM_NAME, DS.MAX_STOCK_LEVEL, DS.SAFETY_STOCK_LEVEL,
                    DS.INT_STORE_LOC_CODE, DS.EXT_STORE_LOC_CODE, DS.ISSUE_UOM_NAME,
                    DS.ABC_IND, DS.PROFIT_CENTER_NUM, DS.CREATED_ON_DT, DS.UNSPSC_CODE,
                    DS.INV_PROD_CAT1, DS.INV_PROD_CAT3, DS.INV_PROD_CAT2,
                    DS.COMMODITY_UOM_NAME, DS.INV_PROD_CAT5, DS.INV_PROD_CAT4,
                    DS.MFG_UOM_NAME, DS.INV_PROD_CAT7, DS.INV_PROD_CAT6,
                    DS.INV_PROD_CAT9, DS.INV_PROD_CAT8, DS.AUX4_CHANGED_ON_DT,
                    DS.LOT_SIZE_NAME, DS.TENANT_ID, DS.INVOICEABLE_ITEM_FLAG,
                    DS.INVENTORY_ORG_ID, DS.SRC_EFF_FROM_DT, DS.INTEGRATION_ID,
                    DS.ISSUE_UOM_CODE, DS.DELETE_FLG, DS.MRP_TYPE_CODE, DS.ACTIVE_FLG,
                    DS.MRP_TIME_FENCE, DS.QA_INSPECT_IND, DS.MFG_UOM_CODE,
                    DS.LOADING_TYPE_NAME, DS.MRP_PROFILE_NAME,
                    PC1.ROW_WID AS INV_PROD_CAT1_ROW_WID,
                    PC2.ROW_WID AS INV_PROD_CAT2_ROW_WID,
                    PC3.ROW_WID AS INV_PROD_CAT3_ROW_WID,
                    PC4.ROW_WID AS INV_PROD_CAT4_ROW_WID,
                    PC5.ROW_WID AS INV_PROD_CAT5_ROW_WID,
                    PC6.ROW_WID AS INV_PROD_CAT6_ROW_WID,
                    PC7.ROW_WID AS INV_PROD_CAT7_ROW_WID,
                    PC8.ROW_WID AS INV_PROD_CAT8_ROW_WID,
                    PC_UNSPSC.ROW_WID AS INV_PROD_CAT_UNSPSC_ROW_WID,
                    PC9.ROW_WID AS INV_PROD_CAT9_ROW_WID,
                    PC10.ROW_WID AS INV_PROD_CAT10_ROW_WID,
                    DS.CUMULATIVE_TOTAL_LEAD_TIME, DS.FIXED_LEAD_TIME,
                    DS.W_STATUS_CODE, DS.PREPROCESSING_LEAD_TIME, DS.STATUS_CODE,
                    DS.VARIABLE_LEAD_TIME, DS.MAKE_BUY_IND,
                    DS.PROCESS_QUALITY_ENABLED_FLG, DS.PRODUCT_TYPE_CODE,
                    DS.X_PRICE_SEQUENCE, DS.X_ORGANIZATION_NAME, DS.X_PRODUCT_DESC,
                    DS.X_UOM_DESC, DS.X_INV_ITEM_FLG, DS.X_STOCK_ITEM_FLG,
                    DS.X_TRANS_FLG, DS.X_REV_FLG, DS.X_COST_FLG,
                    DS.X_GCOA_ACCT, DS.X_GCOA_PROD, DS.X_TAX_CAT,
                    DS.X_GCOA_LOC_ACCT
                FROM workspace.prxbi_dw.w_inventory_product_ds DS
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC1
                    ON DS.INV_PROD_CAT1 = PC1.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC1.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC2
                    ON DS.INV_PROD_CAT2 = PC2.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC2.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC3
                    ON DS.INV_PROD_CAT3 = PC3.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC3.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC4
                    ON DS.INV_PROD_CAT4 = PC4.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC4.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC5
                    ON DS.INV_PROD_CAT5 = PC5.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC5.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC6
                    ON DS.INV_PROD_CAT6 = PC6.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC6.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC7
                    ON DS.INV_PROD_CAT7 = PC7.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC7.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC8
                    ON DS.INV_PROD_CAT8 = PC8.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC8.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC9
                    ON DS.INV_PROD_CAT9 = PC9.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC9.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC10
                    ON DS.INV_PROD_CAT10 = PC10.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC10.DATASOURCE_NUM_ID
                LEFT OUTER JOIN workspace.prxbi_dw.w_prod_cat_dh PC_UNSPSC
                    ON DS.UNSPSC_CODE = PC_UNSPSC.INTEGRATION_ID AND DS.DATASOURCE_NUM_ID = PC_UNSPSC.DATASOURCE_NUM_ID
                WHERE (1=1)
            ) SQ_W
            LEFT OUTER JOIN (
                SELECT BL.DATASOURCE_NUM_ID, BL.ROW_WID, BL.INTEGRATION_ID,
                       BL.EFFECTIVE_TO_DT, BL.EFFECTIVE_FROM_DT
                FROM workspace.prxbi_dw.w_busn_location_d BL
                WHERE (1=1)
            ) LKP_BL
                ON SQ_W.DATASOURCE_NUM_ID = LKP_BL.DATASOURCE_NUM_ID
                AND SQ_W.PLANT_LOC_ID = LKP_BL.INTEGRATION_ID
                AND SQ_W.CREATED_ON_DT >= LKP_BL.EFFECTIVE_FROM_DT
                AND SQ_W.CREATED_ON_DT < LKP_BL.EFFECTIVE_TO_DT
            LEFT OUTER JOIN workspace.prxbi_dw.w_int_org_d LKP_IO
                ON SQ_W.DATASOURCE_NUM_ID = LKP_IO.DATASOURCE_NUM_ID
                AND SQ_W.INVENTORY_ORG_ID = LKP_IO.INTEGRATION_ID
                AND SQ_W.CREATED_ON_DT >= LKP_IO.EFFECTIVE_FROM_DT
                AND SQ_W.CREATED_ON_DT < LKP_IO.EFFECTIVE_TO_DT
            LEFT OUTER JOIN workspace.prxbi_dw.w_product_d LKP_PD
                ON SQ_W.DATASOURCE_NUM_ID = LKP_PD.DATASOURCE_NUM_ID
                AND SQ_W.PRODUCT_ID = LKP_PD.INTEGRATION_ID
                AND SQ_W.CREATED_ON_DT >= LKP_PD.EFFECTIVE_FROM_DT
                AND SQ_W.CREATED_ON_DT < LKP_PD.EFFECTIVE_TO_DT
            LEFT OUTER JOIN (
                SELECT UC.DATASOURCE_NUM_ID, UC.ROW_WID, UC.INTEGRATION_ID,
                       UC.EFFECTIVE_TO_DT, UC.EFFECTIVE_FROM_DT
                FROM workspace.prxbi_dw.w_user_d UC
                WHERE UC.DELETE_FLG = 'N'
            ) LKP_UC
                ON SQ_W.DATASOURCE_NUM_ID = LKP_UC.DATASOURCE_NUM_ID
                AND SQ_W.CHANGED_BY_ID = LKP_UC.INTEGRATION_ID
                AND SQ_W.CHANGED_ON_DT >= LKP_UC.EFFECTIVE_FROM_DT
                AND SQ_W.CHANGED_ON_DT < LKP_UC.EFFECTIVE_TO_DT
    ) INLINE_VIEW
    LEFT OUTER JOIN (
        SELECT UCR.DATASOURCE_NUM_ID, UCR.ROW_WID, UCR.INTEGRATION_ID,
               UCR.EFFECTIVE_TO_DT, UCR.EFFECTIVE_FROM_DT
        FROM workspace.prxbi_dw.w_user_d UCR
        WHERE UCR.DELETE_FLG = 'N'
    ) LKP_W_USER_D_LKP_W_USER_D_CR_1
        ON INLINE_VIEW.DATASOURCE_NUM_ID = LKP_W_USER_D_LKP_W_USER_D_CR_1.DATASOURCE_NUM_ID
        AND INLINE_VIEW.CREATED_BY_ID = LKP_W_USER_D_LKP_W_USER_D_CR_1.INTEGRATION_ID
        AND INLINE_VIEW.CREATED_ON_DT >= LKP_W_USER_D_LKP_W_USER_D_CR_1.EFFECTIVE_FROM_DT
        AND INLINE_VIEW.CREATED_ON_DT < LKP_W_USER_D_LKP_W_USER_D_CR_1.EFFECTIVE_TO_DT
    WHERE (1=1)
) C
LEFT OUTER JOIN workspace.prxbi_dw.w_inventory_product_d T
    ON C.SRC_EFF_FROM_DT = T.SRC_EFF_FROM_DT
    AND C.DATASOURCE_NUM_ID = T.DATASOURCE_NUM_ID
    AND C.INTEGRATION_ID = T.INTEGRATION_ID;

In [ ]:
-- MAGIC %sql
-- Flow table record count
SELECT COUNT(*) AS flow_total_count,
       SUM(CASE
WHEN IND_UPDATE = 'I'
THEN 1 ELSE 0 END) AS insert_count,
       SUM(CASE
WHEN IND_UPDATE = 'U'
THEN 1 ELSE 0 END) AS update_count,
       SUM(CASE
WHEN IND_UPDATE = 'N'
THEN 1 ELSE 0 END) AS no_change_count
FROM workspace.prxbi_dw.i_w_inventory_product_d_flow;

## SCEN_TASK_NO {150} — Optimize Flow Table (replaces CREATE INDEX)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {150}:
Optimize flow table (replaces
CREATE INDEX
on UK columns)
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_w_inventory_product_d_flow
ZORDER BY (SRC_EFF_FROM_DT, DATASOURCE_NUM_ID, INTEGRATION_ID);

## SCEN_TASK_NO {170}-{310} — Bypassed Steps

SCEN_TASK_NO {170}, {180}: IND_UPDATE flagging skipped (Detection Strategy: OUTER — already computed in INSERT)

SCEN_TASK_NO {190}-{310}: Error logging, PK violation detection, autocorrect — all bypassed (Error Logging not enabled)

## SCEN_TASK_NO {330} + {350} — MERGE into Target (UPDATE + INSERT combined)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {330} + {350}: Combined UPDATE (where IND_UPDATE='U') and INSERT (where IND_UPDATE='I')
-- Converted: Oracle tuple-SET UPDATE + separate INSERT -> single MERGE
-- Converted: SYSDATE -> current_timestamp(), TO_DATE -> to_date, SEQUENCE.NEXTVAL -> GENERATED ALWAYS AS IDENTITY (excluded from MERGE)
-- ROW_WID is assumed GENERATED ALWAYS AS IDENTITY on target — excluded from all column lists
MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T
USING workspace.prxbi_dw.i_w_inventory_product_d_flow AS S
ON T.SRC_EFF_FROM_DT = S.SRC_EFF_FROM_DT
   AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
   AND T.INTEGRATION_ID = S.INTEGRATION_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.PRODUCT_WID = S.PRODUCT_WID,
    T.INVENTORY_ORG_WID = S.INVENTORY_ORG_WID,
    T.PLANT_LOC_WID = S.PLANT_LOC_WID,
    T.PRODUCT_NUM = S.PRODUCT_NUM,
    T.ABC_IND = S.ABC_IND,
    T.PLANNER_CODE = S.PLANNER_CODE,
    T.PROCUREMENT_TYPE_CODE = S.PROCUREMENT_TYPE_CODE,
    T.SPC_PROC_TYPE_CODE = S.SPC_PROC_TYPE_CODE,
    T.BUYER_CODE = S.BUYER_CODE,
    T.BUYER_NAME = S.BUYER_NAME,
    T.COMMODITY_CODE = S.COMMODITY_CODE,
    T.COMMODITY_UOM_CODE = S.COMMODITY_UOM_CODE,
    T.PROFIT_CENTER_NUM = S.PROFIT_CENTER_NUM,
    T.REORDER_POINT = S.REORDER_POINT,
    T.SAFETY_STOCK_LEVEL = S.SAFETY_STOCK_LEVEL,
    T.MIN_LOT_SIZE = S.MIN_LOT_SIZE,
    T.MAX_LOT_SIZE = S.MAX_LOT_SIZE,
    T.FIXED_LOT_SIZE = S.FIXED_LOT_SIZE,
    T.MAX_STOCK_LEVEL = S.MAX_STOCK_LEVEL,
    T.LOT_ORDERING_COST = S.LOT_ORDERING_COST,
    T.MRP_TIME_FENCE = S.MRP_TIME_FENCE,
    T.EXT_PROCURE_TIME = S.EXT_PROCURE_TIME,
    T.INTERNAL_MFG_TIME = S.INTERNAL_MFG_TIME,
    T.MAX_STORAGE_DAYS = S.MAX_STORAGE_DAYS,
    T.MRP_PROFILE_CODE = S.MRP_PROFILE_CODE,
    T.MRP_TYPE_CODE = S.MRP_TYPE_CODE,
    T.MRP_GRP_CODE = S.MRP_GRP_CODE,
    T.LOT_SIZE_CODE = S.LOT_SIZE_CODE,
    T.BACKFLUSH_IND = S.BACKFLUSH_IND,
    T.QA_INSPECT_IND = S.QA_INSPECT_IND,
    T.REPETITIVE_MFG_IND = S.REPETITIVE_MFG_IND,
    T.BULK_ITEM_IND = S.BULK_ITEM_IND,
    T.FORECAST_PERIOD = S.FORECAST_PERIOD,
    T.MFG_UOM_CODE = S.MFG_UOM_CODE,
    T.ISSUE_UOM_CODE = S.ISSUE_UOM_CODE,
    T.MANUFACTURING_PLACE = S.MANUFACTURING_PLACE,
    T.LOADING_TYPE_CODE = S.LOADING_TYPE_CODE,
    T.INT_STORE_LOC_CODE = S.INT_STORE_LOC_CODE,
    T.EXT_STORE_LOC_CODE = S.EXT_STORE_LOC_CODE,
    T.ACTIVE_FLG = S.ACTIVE_FLG,
    T.CREATED_BY_WID = S.CREATED_BY_WID,
    T.CHANGED_BY_WID = S.CHANGED_BY_WID,
    T.CREATED_ON_DT = S.CREATED_ON_DT,
    T.CHANGED_ON_DT = S.CHANGED_ON_DT,
    T.AUX1_CHANGED_ON_DT = S.AUX1_CHANGED_ON_DT,
    T.AUX2_CHANGED_ON_DT = S.AUX2_CHANGED_ON_DT,
    T.AUX3_CHANGED_ON_DT = S.AUX3_CHANGED_ON_DT,
    T.AUX4_CHANGED_ON_DT = S.AUX4_CHANGED_ON_DT,
    T.SRC_EFF_TO_DT = S.SRC_EFF_TO_DT,
    T.EFFECTIVE_FROM_DT = S.EFFECTIVE_FROM_DT,
    T.DELETE_FLG = S.DELETE_FLG,
    T.TENANT_ID = S.TENANT_ID,
    T.X_CUSTOM = S.X_CUSTOM,
    T.INV_PROD_CAT1 = S.INV_PROD_CAT1,
    T.INV_PROD_CAT2 = S.INV_PROD_CAT2,
    T.INV_PROD_CAT3 = S.INV_PROD_CAT3,
    T.INV_PROD_CAT4 = S.INV_PROD_CAT4,
    T.INV_PROD_CAT5 = S.INV_PROD_CAT5,
    T.INV_PROD_CAT6 = S.INV_PROD_CAT6,
    T.INV_PROD_CAT7 = S.INV_PROD_CAT7,
    T.INV_PROD_CAT8 = S.INV_PROD_CAT8,
    T.INV_PROD_CAT9 = S.INV_PROD_CAT9,
    T.INV_PROD_CAT10 = S.INV_PROD_CAT10,
    T.INV_PROD_CAT1_WID = S.INV_PROD_CAT1_WID,
    T.INV_PROD_CAT2_WID = S.INV_PROD_CAT2_WID,
    T.INV_PROD_CAT3_WID = S.INV_PROD_CAT3_WID,
    T.INV_PROD_CAT4_WID = S.INV_PROD_CAT4_WID,
    T.INV_PROD_CAT5_WID = S.INV_PROD_CAT5_WID,
    T.INV_PROD_CAT6_WID = S.INV_PROD_CAT6_WID,
    T.INV_PROD_CAT7_WID = S.INV_PROD_CAT7_WID,
    T.INV_PROD_CAT8_WID = S.INV_PROD_CAT8_WID,
    T.INV_PROD_CAT9_WID = S.INV_PROD_CAT9_WID,
    T.INV_PROD_CAT10_WID = S.INV_PROD_CAT10_WID,
    T.INVOICEABLE_ITEM_FLAG = S.INVOICEABLE_ITEM_FLAG,
    T.INVOICE_ENABLED_FLAG = S.INVOICE_ENABLED_FLAG,
    T.PRIMARY_UOM_CODE = S.PRIMARY_UOM_CODE,
    T.C_PRIMARY_UOM_CODE = S.C_PRIMARY_UOM_CODE,
    T.UNSPSC_CODE = S.UNSPSC_CODE,
    T.UNSPSC_INV_PROD_CAT_WID = S.UNSPSC_INV_PROD_CAT_WID,
    T.COMMODITY_NAME = S.COMMODITY_NAME,
    T.COMMODITY_UOM_NAME = S.COMMODITY_UOM_NAME,
    T.EXT_STORE_LOC_NAME = S.EXT_STORE_LOC_NAME,
    T.INT_STORE_LOC_NAME = S.INT_STORE_LOC_NAME,
    T.ISSUE_UOM_NAME = S.ISSUE_UOM_NAME,
    T.LOADING_TYPE_NAME = S.LOADING_TYPE_NAME,
    T.LOT_SIZE_NAME = S.LOT_SIZE_NAME,
    T.MFG_UOM_NAME = S.MFG_UOM_NAME,
    T.MRP_GRP_NAME = S.MRP_GRP_NAME,
    T.MRP_PROFILE_NAME = S.MRP_PROFILE_NAME,
    T.MRP_TYPE_NAME = S.MRP_TYPE_NAME,
    T.PLANNER_NAME = S.PLANNER_NAME,
    T.PRIMARY_UOM_NAME = S.PRIMARY_UOM_NAME,
    T.PROCUREMENT_TYPE_NAME = S.PROCUREMENT_TYPE_NAME,
    T.PROFIT_CENTER_NAME = S.PROFIT_CENTER_NAME,
    T.SPC_PROC_TYPE_NAME = S.SPC_PROC_TYPE_NAME,
    T.STATUS_CODE = S.STATUS_CODE,
    T.W_STATUS_CODE = S.W_STATUS_CODE,
    T.PRODUCT_TYPE_CODE = S.PRODUCT_TYPE_CODE,
    T.MAKE_BUY_IND = S.MAKE_BUY_IND,
    T.FIXED_LEAD_TIME = S.FIXED_LEAD_TIME,
    T.VARIABLE_LEAD_TIME = S.VARIABLE_LEAD_TIME,
    T.CUMULATIVE_TOTAL_LEAD_TIME = S.CUMULATIVE_TOTAL_LEAD_TIME,
    T.POSTPROCESSING_LEAD_TIME = S.POSTPROCESSING_LEAD_TIME,
    T.PREPROCESSING_LEAD_TIME = S.PREPROCESSING_LEAD_TIME,
    T.PROCESS_QUALITY_ENABLED_FLG = S.PROCESS_QUALITY_ENABLED_FLG,
    T.X_PRICE_SEQUENCE = S.X_PRICE_SEQUENCE,
    T.X_ORGANIZATION_NAME = S.X_ORGANIZATION_NAME,
    T.X_PRODUCT_DESC = S.X_PRODUCT_DESC,
    T.X_UOM_DESC = S.X_UOM_DESC,
    T.X_INV_ITEM_FLG = S.X_INV_ITEM_FLG,
    T.X_STOCK_ITEM_FLG = S.X_STOCK_ITEM_FLG,
    T.X_TRANS_FLG = S.X_TRANS_FLG,
    T.X_REV_FLG = S.X_REV_FLG,
    T.X_COST_FLG = S.X_COST_FLG,
    T.X_GCOA_ACCT = S.X_GCOA_ACCT,
    T.X_GCOA_PROD = S.X_GCOA_PROD,
    T.X_TAX_CAT = S.X_TAX_CAT,
    T.ORGANIZATION_ID = S.ORGANIZATION_ID,
    T.X_GCOA_LOC_ACCT = S.X_GCOA_LOC_ACCT,
    T.EFFECTIVE_TO_DT = to_date('01/01/3714 00:00:00', 'MM/dd/yyyy HH:mm:ss'),
    T.CURRENT_FLG = 'Y',
    T.W_UPDATE_DT = current_timestamp(),
    T.ETL_PROC_WID = ${ETL_PROC_WID}
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    PRODUCT_WID,
    INVENTORY_ORG_WID,
    PLANT_LOC_WID,
    PRODUCT_NUM,
    ABC_IND,
    PLANNER_CODE,
    PROCUREMENT_TYPE_CODE,
    SPC_PROC_TYPE_CODE,
    BUYER_CODE,
    BUYER_NAME,
    COMMODITY_CODE,
    COMMODITY_UOM_CODE,
    PROFIT_CENTER_NUM,
    REORDER_POINT,
    SAFETY_STOCK_LEVEL,
    MIN_LOT_SIZE,
    MAX_LOT_SIZE,
    FIXED_LOT_SIZE,
    MAX_STOCK_LEVEL,
    LOT_ORDERING_COST,
    MRP_TIME_FENCE,
    EXT_PROCURE_TIME,
    INTERNAL_MFG_TIME,
    MAX_STORAGE_DAYS,
    MRP_PROFILE_CODE,
    MRP_TYPE_CODE,
    MRP_GRP_CODE,
    LOT_SIZE_CODE,
    BACKFLUSH_IND,
    QA_INSPECT_IND,
    REPETITIVE_MFG_IND,
    BULK_ITEM_IND,
    FORECAST_PERIOD,
    MFG_UOM_CODE,
    ISSUE_UOM_CODE,
    MANUFACTURING_PLACE,
    LOADING_TYPE_CODE,
    INT_STORE_LOC_CODE,
    EXT_STORE_LOC_CODE,
    ACTIVE_FLG,
    CREATED_BY_WID,
    CHANGED_BY_WID,
    CREATED_ON_DT,
    CHANGED_ON_DT,
    AUX1_CHANGED_ON_DT,
    AUX2_CHANGED_ON_DT,
    AUX3_CHANGED_ON_DT,
    AUX4_CHANGED_ON_DT,
    SRC_EFF_FROM_DT,
    SRC_EFF_TO_DT,
    EFFECTIVE_FROM_DT,
    DELETE_FLG,
    DATASOURCE_NUM_ID,
    INTEGRATION_ID,
    TENANT_ID,
    X_CUSTOM,
    INV_PROD_CAT1,
    INV_PROD_CAT2,
    INV_PROD_CAT3,
    INV_PROD_CAT4,
    INV_PROD_CAT5,
    INV_PROD_CAT6,
    INV_PROD_CAT7,
    INV_PROD_CAT8,
    INV_PROD_CAT9,
    INV_PROD_CAT10,
    INV_PROD_CAT1_WID,
    INV_PROD_CAT2_WID,
    INV_PROD_CAT3_WID,
    INV_PROD_CAT4_WID,
    INV_PROD_CAT5_WID,
    INV_PROD_CAT6_WID,
    INV_PROD_CAT7_WID,
    INV_PROD_CAT8_WID,
    INV_PROD_CAT9_WID,
    INV_PROD_CAT10_WID,
    INVOICEABLE_ITEM_FLAG,
    INVOICE_ENABLED_FLAG,
    PRIMARY_UOM_CODE,
    C_PRIMARY_UOM_CODE,
    UNSPSC_CODE,
    UNSPSC_INV_PROD_CAT_WID,
    COMMODITY_NAME,
    COMMODITY_UOM_NAME,
    EXT_STORE_LOC_NAME,
    INT_STORE_LOC_NAME,
    ISSUE_UOM_NAME,
    LOADING_TYPE_NAME,
    LOT_SIZE_NAME,
    MFG_UOM_NAME,
    MRP_GRP_NAME,
    MRP_PROFILE_NAME,
    MRP_TYPE_NAME,
    PLANNER_NAME,
    PRIMARY_UOM_NAME,
    PROCUREMENT_TYPE_NAME,
    PROFIT_CENTER_NAME,
    SPC_PROC_TYPE_NAME,
    STATUS_CODE,
    W_STATUS_CODE,
    PRODUCT_TYPE_CODE,
    MAKE_BUY_IND,
    FIXED_LEAD_TIME,
    VARIABLE_LEAD_TIME,
    CUMULATIVE_TOTAL_LEAD_TIME,
    POSTPROCESSING_LEAD_TIME,
    PREPROCESSING_LEAD_TIME,
    PROCESS_QUALITY_ENABLED_FLG,
    X_PRICE_SEQUENCE,
    X_ORGANIZATION_NAME,
    X_PRODUCT_DESC,
    X_UOM_DESC,
    X_INV_ITEM_FLG,
    X_STOCK_ITEM_FLG,
    X_TRANS_FLG,
    X_REV_FLG,
    X_COST_FLG,
    X_GCOA_ACCT,
    X_GCOA_PROD,
    X_TAX_CAT,
    ORGANIZATION_ID,
    X_GCOA_LOC_ACCT,
    W_INSERT_DT,
    W_UPDATE_DT,
    ETL_PROC_WID,
    CURRENT_FLG,
    EFFECTIVE_TO_DT
) VALUES (
    S.PRODUCT_WID,
    S.INVENTORY_ORG_WID,
    S.PLANT_LOC_WID,
    S.PRODUCT_NUM,
    S.ABC_IND,
    S.PLANNER_CODE,
    S.PROCUREMENT_TYPE_CODE,
    S.SPC_PROC_TYPE_CODE,
    S.BUYER_CODE,
    S.BUYER_NAME,
    S.COMMODITY_CODE,
    S.COMMODITY_UOM_CODE,
    S.PROFIT_CENTER_NUM,
    S.REORDER_POINT,
    S.SAFETY_STOCK_LEVEL,
    S.MIN_LOT_SIZE,
    S.MAX_LOT_SIZE,
    S.FIXED_LOT_SIZE,
    S.MAX_STOCK_LEVEL,
    S.LOT_ORDERING_COST,
    S.MRP_TIME_FENCE,
    S.EXT_PROCURE_TIME,
    S.INTERNAL_MFG_TIME,
    S.MAX_STORAGE_DAYS,
    S.MRP_PROFILE_CODE,
    S.MRP_TYPE_CODE,
    S.MRP_GRP_CODE,
    S.LOT_SIZE_CODE,
    S.BACKFLUSH_IND,
    S.QA_INSPECT_IND,
    S.REPETITIVE_MFG_IND,
    S.BULK_ITEM_IND,
    S.FORECAST_PERIOD,
    S.MFG_UOM_CODE,
    S.ISSUE_UOM_CODE,
    S.MANUFACTURING_PLACE,
    S.LOADING_TYPE_CODE,
    S.INT_STORE_LOC_CODE,
    S.EXT_STORE_LOC_CODE,
    S.ACTIVE_FLG,
    S.CREATED_BY_WID,
    S.CHANGED_BY_WID,
    S.CREATED_ON_DT,
    S.CHANGED_ON_DT,
    S.AUX1_CHANGED_ON_DT,
    S.AUX2_CHANGED_ON_DT,
    S.AUX3_CHANGED_ON_DT,
    S.AUX4_CHANGED_ON_DT,
    S.SRC_EFF_FROM_DT,
    S.SRC_EFF_TO_DT,
    S.EFFECTIVE_FROM_DT,
    S.DELETE_FLG,
    S.DATASOURCE_NUM_ID,
    S.INTEGRATION_ID,
    S.TENANT_ID,
    S.X_CUSTOM,
    S.INV_PROD_CAT1,
    S.INV_PROD_CAT2,
    S.INV_PROD_CAT3,
    S.INV_PROD_CAT4,
    S.INV_PROD_CAT5,
    S.INV_PROD_CAT6,
    S.INV_PROD_CAT7,
    S.INV_PROD_CAT8,
    S.INV_PROD_CAT9,
    S.INV_PROD_CAT10,
    S.INV_PROD_CAT1_WID,
    S.INV_PROD_CAT2_WID,
    S.INV_PROD_CAT3_WID,
    S.INV_PROD_CAT4_WID,
    S.INV_PROD_CAT5_WID,
    S.INV_PROD_CAT6_WID,
    S.INV_PROD_CAT7_WID,
    S.INV_PROD_CAT8_WID,
    S.INV_PROD_CAT9_WID,
    S.INV_PROD_CAT10_WID,
    S.INVOICEABLE_ITEM_FLAG,
    S.INVOICE_ENABLED_FLAG,
    S.PRIMARY_UOM_CODE,
    S.C_PRIMARY_UOM_CODE,
    S.UNSPSC_CODE,
    S.UNSPSC_INV_PROD_CAT_WID,
    S.COMMODITY_NAME,
    S.COMMODITY_UOM_NAME,
    S.EXT_STORE_LOC_NAME,
    S.INT_STORE_LOC_NAME,
    S.ISSUE_UOM_NAME,
    S.LOADING_TYPE_NAME,
    S.LOT_SIZE_NAME,
    S.MFG_UOM_NAME,
    S.MRP_GRP_NAME,
    S.MRP_PROFILE_NAME,
    S.MRP_TYPE_NAME,
    S.PLANNER_NAME,
    S.PRIMARY_UOM_NAME,
    S.PROCUREMENT_TYPE_NAME,
    S.PROFIT_CENTER_NAME,
    S.SPC_PROC_TYPE_NAME,
    S.STATUS_CODE,
    S.W_STATUS_CODE,
    S.PRODUCT_TYPE_CODE,
    S.MAKE_BUY_IND,
    S.FIXED_LEAD_TIME,
    S.VARIABLE_LEAD_TIME,
    S.CUMULATIVE_TOTAL_LEAD_TIME,
    S.POSTPROCESSING_LEAD_TIME,
    S.PREPROCESSING_LEAD_TIME,
    S.PROCESS_QUALITY_ENABLED_FLG,
    S.X_PRICE_SEQUENCE,
    S.X_ORGANIZATION_NAME,
    S.X_PRODUCT_DESC,
    S.X_UOM_DESC,
    S.X_INV_ITEM_FLG,
    S.X_STOCK_ITEM_FLG,
    S.X_TRANS_FLG,
    S.X_REV_FLG,
    S.X_COST_FLG,
    S.X_GCOA_ACCT,
    S.X_GCOA_PROD,
    S.X_TAX_CAT,
    S.ORGANIZATION_ID,
    S.X_GCOA_LOC_ACCT,
    current_timestamp(),
    current_timestamp(),
    ${ETL_PROC_WID},
    S.CURRENT_FLG,
    S.EFFECTIVE_TO_DT
);

## SCEN_TASK_NO {360} — Update ETL Load Dates

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {360}: Update ETL load dates reference table
-- Converted: DECODE -> CASE WHEN, SYSDATE -> current_timestamp()
MERGE INTO workspace.prxbi_dw.w_etl_load_dates AS T
USING (
    SELECT
        ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
        'SILOS_SIL_INVENTORYPRODUCTDIMENSION' AS PACKAGE_NAME,
        '${ETL_USAGE_CODE}' AS ETL_USAGE_CODE
) AS S
ON T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
   AND T.PACKAGE_NAME = S.PACKAGE_NAME
   AND T.ETL_USAGE_CODE = S.ETL_USAGE_CODE
WHEN MATCHED THEN UPDATE SET
    T.TARGET_TABLE_NAME = 'W_INVENTORY_PRODUCT_D',
    T.ETL_PROC_WID = ${ETL_PROC_WID},
    T.LOAD_PLAN_ID = '${EXECUTION_ID}',
    T.WIP_LOAD_START_DATE = date_add(current_timestamp(), -1 * ${PRUNE_DAYS}),
    T.ETL_LOAD_DATE = current_timestamp(),
    T.COMMITTED = CASE WHEN '${IS_INCREMENTAL}' = 'Y' THEN '1' ELSE '0' END;

## SCEN_TASK_NO {370} — Insert ETL Load Dates Log

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {370}: Insert into ETL load dates history/log table
INSERT INTO workspace.prxbi_dw.w_etl_load_dates_log
(
    DATASOURCE_NUM_ID,
    PACKAGE_NAME,
    TARGET_TABLE_NAME,
    ETL_USAGE_CODE,
    ETL_PROC_WID,
    LOAD_PLAN_ID,
    SESSION_ID,
    WIP_LOAD_START_DATE,
    LAST_MAX_DATE,
    ETL_LOAD_DATE,
    COMMITTED
)
SELECT
    DATASOURCE_NUM_ID,
    PACKAGE_NAME,
    TARGET_TABLE_NAME,
    ETL_USAGE_CODE,
    ETL_PROC_WID,
    LOAD_PLAN_ID,
    3260538 AS SESSION_ID,
    WIP_LOAD_START_DATE,
    LAST_MAX_DATE,
    ETL_LOAD_DATE,
    COMMITTED
FROM workspace.prxbi_dw.w_etl_load_dates
WHERE DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID}
  AND PACKAGE_NAME = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'
  AND ETL_USAGE_CODE = '${ETL_USAGE_CODE}';

## Optimize Target Table

In [ ]:
-- MAGIC %sql
--
Optimize target table
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.w_inventory_product_d
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## SCEN_TASK_NO {420} + {440} — Cleanup

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {420}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_w_inventory_product_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {440}:
Drop error table if empty
-- In Spark we conditionally drop;

if rows exist they are preserved
DROP TABLE IF EXISTS workspace.prxbi_dw.e_3260538_1;

## Validation

In [ ]:
-- MAGIC %sql
-- Final record count on target
SELECT COUNT(*) AS total_records
FROM workspace.prxbi_dw.w_inventory_product_d;

In [ ]:
-- MAGIC %sql
-- Sample recently updated records
SELECT *
FROM workspace.prxbi_dw.w_inventory_product_d
WHERE W_UPDATE_DT >= date_add(current_timestamp(), -1)
LIMIT 20;

## Conversion Notes

### Key conversions applied:
- `NVL()` → `COALESCE()`
- `DECODE()` → `CASE WHEN`
- `SYSDATE` → `current_timestamp()`
- `a || '~' || b` → `CONCAT(a, '~', b)`
- `TO_DATE('MM/DD/YYYY HH24:MI:SS')` → `to_date('MM/dd/yyyy HH:mm:ss')`
- `SUBSTR(s,p,l)` → `substring(s,p,l)`
- `VARCHAR2` → `STRING`, `NUMBER(10,0)` → `BIGINT`, `NUMBER` → `DOUBLE`, `DATE` → `TIMESTAMP`, `CHAR(1)` → `STRING`, `UROWID` → `STRING`
- `NOLOGGING` / `/*+ append */` → removed
- `DROP TABLE ... PURGE` → `DROP TABLE IF EXISTS`
- `BEGIN...END` PL/SQL blocks → removed
- `COMMIT` → removed (implicit)
- `CREATE INDEX` → `OPTIMIZE ... ZORDER BY`
- Oracle tuple-SET `UPDATE ... SET (cols) = (SELECT ...)` + separate `INSERT ... WHERE NOT EXISTS` → single `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED`
- `W_INVENTORY_PRODUCT_D_SEQ.NEXTVAL` for `ROW_WID` → excluded from MERGE (assumed `GENERATED ALWAYS AS IDENTITY` on target)
- `T.ROWID IS NOT NULL` change detection → `T.INTEGRATION_ID IS NOT NULL` (checks if target row exists via LEFT JOIN)
- `#BIAPPS.*` parameters → `${...}` widget references

### Manual actions required:
1. **Verify ROW_WID identity**: Confirm that `workspace.prxbi_dw.w_inventory_product_d.ROW_WID` is defined as `GENERATED ALWAYS AS IDENTITY`. If not, adjust the MERGE to include ROW_WID.
2. **Verify widget parameter values**: Ensure all widget parameters (`SOURCE_CODE`, `TARGET_CODE`, `LOW_DATE`, etc.) are provided at runtime.
3. **Domain member map subqueries**: The correlated scalar subqueries for `C_PRIMARY_UOM_CODE` should be validated — they may need to be refactored to joins if performance is poor.
4. **SCEN_TASK_NO {340}**: Records are not deleted when running in UPDATE mode — no DELETE step was generated.